In [1]:
import os 
import numpy as np 

In [2]:

os.chdir(r"C:\End-To-End-MLOPS-Data-Science-Project-Implementation-With-Deployment")
print(os.getcwd())
os.getcwd()

C:\End-To-End-MLOPS-Data-Science-Project-Implementation-With-Deployment


'C:\\End-To-End-MLOPS-Data-Science-Project-Implementation-With-Deployment'

In [3]:
# %pwd

In [4]:
from dataclasses import dataclass
from pathlib import Path 

In [5]:
@dataclass(frozen=True)
class DataIngestionConfig:  #return type of entity
    root_dir:Path
    source_URL: str
    local_data_file: Path
    unzip_dir: Path

In [6]:
# import sys
# import os

# sys.path.append(os.getcwd())

In [7]:
from src.ml_deployment.constants import *


In [8]:
# !pip install ensure

In [9]:
import src.ml_deployment.utils.common
from src.ml_deployment.utils.common import read_yaml, create_directories

In [14]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath=CONFIG_FILE_PATH,
        params_filepath=PARAMS_FILE_PATH,
        schema_filepath=SCHEMA_FILE_PATH
    ):
        self.config_filepath = config_filepath
        self.params_filepath = params_filepath
        self.schema_filepath = schema_filepath
        
        self.config=read_yaml(self.config_filepath)
        self.params=read_yaml(self.params_filepath)
        self.schema=read_yaml(self.schema_filepath)
        
        
        create_directories([self.config.artifacts_root])
        
    def get_data_ingestion_config(self) -> DataIngestionConfig:
        config=self.config.data_ingestion
        create_directories([config.root_dir])
        
        data_ingestion_config=DataIngestionConfig(
            root_dir=Path(config.root_dir),
            source_URL=config.source_URL,
            local_data_file=Path(config.local_data_file),
            unzip_dir=Path(config.unzip_dir)
        )
        
        return data_ingestion_config

In [15]:
import os
import urllib.request as request 
import zipfile 
from src.ml_deployment import logger
from src.ml_deployment.utils.common import get_size

In [16]:
class DataIngestion:
    def __init__(self,config: DataIngestionConfig):
        self.config= config 
        
    def download_file(self):
        if not os.path.exists(self.config.local_data_file):
            filename,headers=request.urlretrieve(
                url=self.config.source_URL,
                filename=self.config.local_data_file
            )
            logger.info(f'{filename} download ! with following info: \n{headers}')
                
        else:
            logger.info(f'File already exists of size :{get_size(self.config.local_data_file)}')
        
    def extract_zip_file(self):
    
        unzip_path=self.config.unzip_dir
        os.makedirs(unzip_path,exist_ok=True)
        
        with zipfile.ZipFile(self.config.local_data_file,'r') as zip_ref:
            zip_ref.extractall(unzip_path)
                
        

In [17]:
try:
    config=ConfigurationManager()
    data_ingestion_config=config.get_data_ingestion_config()
    
    data_ingestion=DataIngestion(config=data_ingestion_config)
    data_ingestion.download_file()
    data_ingestion.extract_zip_file()
except Exception as e:
    raise e


[2026-02-21 20:39:59,031: INFO: common: yaml file: C:\End-To-End-MLOPS-Data-Science-Project-Implementation-With-Deployment\config\config.yaml loaded succesfully]
[2026-02-21 20:39:59,033: INFO: common: yaml file: C:\End-To-End-MLOPS-Data-Science-Project-Implementation-With-Deployment\params.yaml loaded succesfully]
[2026-02-21 20:39:59,036: INFO: common: yaml file: C:\End-To-End-MLOPS-Data-Science-Project-Implementation-With-Deployment\schema.yaml loaded succesfully]
[2026-02-21 20:39:59,037: INFO: common: created directory at : artifacts]
[2026-02-21 20:39:59,039: INFO: common: created directory at : artifacts/data_ingestion]
[2026-02-21 20:39:59,040: INFO: 2401083580: File already exists of size :~21 KB]
